# 🛰️ تقسیم‌بندی معنایی با یادگیری عمیق
## Semantic Segmentation with Deep Learning — Potsdam Dataset

---

این نوت‌بوک یک **راهنمای آموزشی کامل** است که هر سه مرحله پروژه را از ابتدا تا انتها پوشش می‌دهد:

| مرحله | عنوان | هدف |
|---|---|---|
| ① | آماده‌سازی داده | بارگذاری، تصویرسازی، تقسیم K-Fold |
| ② | مدل ساده (CNN) | آموزش شبکه‌ای ساده با 4 باند |
| ③ | U-Net | آموزش Encoder-Decoder با 5 باند |

> **💡 نحوه استفاده:** هر سلول را از بالا به پایین اجرا کنید. توضیحات فارسی هر بخش را بخوانید، سپس سلول کد را اجرا کنید.

---
## 📦 مرحله ۰ — نصب کتابخانه‌ها

قبل از هر چیز، کتابخانه‌های مورد نیاز پروژه را نصب می‌کنیم.

- **rasterio**: خواندن فایل‌های GeoTIFF (فرمت داده Potsdam)
- **numpy / matplotlib**: پردازش آرایه و رسم نمودار
- **scikit-learn**: تقسیم K-Fold
- **tensorflow**: ساخت و آموزش مدل‌های یادگیری عمیق


In [ ]:
!pip install rasterio numpy matplotlib scikit-learn tensorflow -q
print('✅ نصب کامل شد!')

---
## ⚙️ تنظیمات اولیه و Import ها

اینجا تمام import‌ها و تنظیمات اصلی پروژه را انجام می‌دهیم.

**مهم‌ترین متغیر:** `DATA_DIR` — مسیر پوشه‌ای که فایل‌های `.tif` در آن قرار دارند.
اگر dataset کامل را دانلود کرده‌اید، این مسیر را به مسیر dataset کامل تغییر دهید.


In [ ]:
import os, random, json
import numpy as np
import matplotlib
matplotlib.use('Agg')  # headless mode
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import ModelCheckpoint
from sklearn.model_selection import KFold
from IPython.display import Image, display

# ── Reproducibility seed ────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Global settings ────────────────────────
N_FOLDS     = 5
NUM_CLASSES = 6

def discover_data_dir():
    possible = [
        os.path.join(os.getcwd(), 'Potsdam-GeoTif'),
        os.path.join(os.getcwd(), 'data'),
        os.path.join(os.getcwd(), 'PROJECT', 'Potsdam-GeoTif'),
        os.path.join(os.getcwd(), 'PROJECT', 'data'),
        os.getcwd()
    ]
    for p in [p for p in possible if os.path.exists(p)]:
        try:
            if any(f.endswith('.tif') for f in os.listdir(p)):
                return p
        except:
            continue
    return 'data'

DATA_DIR = discover_data_dir()
print(f"Data directory: {DATA_DIR}")


---
# 📁 مرحله ۱ — آماده‌سازی Dataset

## درباره Dataset

**2D Semantic Labeling Potsdam** یک dataset هوافضایی از شهر پوتسدام آلمان است.
هر فایل GeoTIFF شامل **۶ باند** است:

| شماره باند | نام | کاربرد |
|---|---|---|
| ۰ | Red (قرمز) | ویژگی ورودی |
| ۱ | Green (سبز) | ویژگی ورودی |
| ۲ | Blue (آبی) | ویژگی ورودی |
| ۳ | Infrared (مادون قرمز) | ویژگی ورودی |
| ۴ | Elevation (ارتفاع) | ویژگی ورودی |
| ۵ | **Labels** (برچسب) | **خروجی هدف** |

وظیفه: به ازای **هر پیکسل** یکی از ۶ کلاس زیر را پیش‌بینی کن:
- 🟨 ماشین • 🟩 درخت • 🟦 ساختمان • ⬜ سطح نفوذناپذیر • 🩵 پوشش گیاهی • 🟥 پس‌زمینه


### ۱.۱ — پیدا کردن فایل‌های GeoTIFF

این سلول تمام فایل‌های `.tif` را در `DATA_DIR` پیدا می‌کند.
با dataset کامل، انتظار داریم بیش از ۱۵,۰۰۰ فایل پیدا شود.


In [ ]:
def get_all_tif_files(data_dir):
    """تمام فایل‌های .tif را در دایرکتوری پیدا می‌کند."""
    tif_files = []
    for root, dirs, files in os.walk(data_dir):
        for f in files:
            if f.endswith('.tif'):
                tif_files.append(os.path.join(root, f))
    return tif_files

all_files = get_all_tif_files(DATA_DIR)
print(f'📂 تعداد فایل‌های GeoTIFF پیدا شده: {len(all_files)}')
for f in all_files[:5]:
    print(f'   • {os.path.basename(f)}')

### ۱.۲ — تصویرسازی یک نمونه

طبق تکلیف، باید یک تایل را انتخاب کرده و سه نمایش زیر را نشان دهیم:
1. **تصویر RGB** — با استفاده از باندهای ۰، ۱، ۲
2. **باند ارتفاع** — با colorbar
3. **نقشه برچسب** — با رنگ‌های کلاس و legend

> 📌 فایل `0000000224-0000042784.tif` فایل نمونه است و از انتخاب تصادفی حذف می‌شود.
> اگر فقط همین فایل موجود باشد، از همان استفاده می‌شود (fallback).


In [ ]:
def normalize_band(band):
    """باند را به بازه [0,1] نرمال می‌کند."""
    b_min, b_max = band.min(), band.max()
    if b_max == b_min:
        return np.zeros_like(band, dtype=np.float32)
    return (band - b_min).astype(np.float32) / (b_max - b_min)

def label_to_rgb(label_band, colors):
    """باند برچسب را به تصویر RGB تبدیل می‌کند."""
    h, w = label_band.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for idx, c in enumerate(colors):
        rgb[label_band == idx] = c
    return rgb

# انتخاب فایل نمونه
EXCLUDED = '0000000224-0000042784.tif'
candidates = [f for f in all_files if EXCLUDED not in f]
sample_file = random.choice(candidates) if candidates else all_files[0]
print('📄 فایل انتخاب شده:', os.path.basename(sample_file))

# خواندن داده
with rasterio.open(sample_file) as src:
    data = src.read()  # (6, H, W)
print('ابعاد داده:', data.shape, ' → (باندها، ارتفاع، عرض)')

# ساخت تصاویر
rgb_img   = np.stack([normalize_band(data[0]), normalize_band(data[1]), normalize_band(data[2])], axis=-1)
elev_img  = normalize_band(data[4].astype(np.float32))
label_rgb = label_to_rgb(data[5].astype(np.int32), CLASS_COLORS)

patches = [mpatches.Patch(color=[c/255 for c in CLASS_COLORS[i]], label=CLASS_NAMES[i])
           for i in range(NUM_CLASSES)]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('تصویرسازی باندهای تایل نمونه', fontsize=14, fontweight='bold')

axes[0].imshow(rgb_img)
axes[0].set_title('باندهای RGB', fontsize=11)
axes[0].axis('off')

im = axes[1].imshow(elev_img, cmap='terrain')
axes[1].set_title('باند ارتفاع', fontsize=11)
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

axes[2].imshow(label_rgb)
axes[2].set_title('نقشه برچسب (Label)', fontsize=11)
axes[2].axis('off')
axes[2].legend(handles=patches, loc='lower right', fontsize=7, framealpha=0.9)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'step1_visualization.png'), dpi=150, bbox_inches='tight')
plt.close()
display(Image(os.path.join(DATA_DIR, 'step1_visualization.png')))
print('✅ تصویر ذخیره شد: step1_visualization.png')

### ۱.۳ — آمار کلاس‌ها در تایل نمونه

درصد پیکسل‌های هر کلاس در تایل انتخاب شده را محاسبه می‌کنیم.
این اطلاعات به ما نشان می‌دهد که dataset چه توزیعی دارد.


In [ ]:
label_band = data[5].astype(np.int32)
total_pixels = label_band.size

print('📊 توزیع کلاس‌ها:')
print(f'{"کلاس":<30} {"تعداد پیکسل":>15} {"درصد":>10}')
print('-' * 58)
for i, name in enumerate(CLASS_NAMES):
    count = (label_band == i).sum()
    pct   = 100 * count / total_pixels
    bar   = '█' * int(pct / 2)
    print(f'{name:<30} {count:>15,} {pct:>9.2f}%  {bar}')

### ۱.۴ — تقسیم‌بندی K-Fold

داده‌ها را به **۵ fold** تقسیم می‌کنیم:
- **Fold 1+2+3** → آموزش (Training)
- **Fold 4** → اعتبارسنجی (Validation)
- **Fold 5** → تست (Test) — کاملاً جدا نگه می‌داریم

این تقسیم‌بندی در فایل `fold_splits.json` ذخیره می‌شود و در مراحل ۲ و ۳ استفاده می‌شود.

> ⚠️ اگر کمتر از ۵ فایل داریم، فایل‌ها را تکرار می‌کنیم تا مکانیزم K-Fold را نشان دهیم.


In [ ]:
# =====================================================================
# =====================================================================
# =====================================================================
# CONFIGURATION - Adjust paths as needed
# =====================================================================
# Original local path:
# DATA_DIR = r'c:\Users\mina_\OneDrive\Documents\DESING_OF_AI_SYSTEMS\Semantic Segmentation with Deep Learning\PROJECT\Potsdam-GeoTif'

import os
def discover_data_dir():
    possible = [
        os.path.join(os.getcwd(), 'Potsdam-GeoTif'),
        os.path.join(os.getcwd(), 'data'),
        os.path.join(os.getcwd(), 'PROJECT', 'Potsdam-GeoTif'),
        os.path.join(os.getcwd(), 'PROJECT', 'data'),
        os.getcwd()
    ]
    existing = [p for p in possible if os.path.exists(p)]
    for p in existing:
        try:
            if any(f.endswith('.tif') for f in os.listdir(p)):
                return p
        except:
            continue
    return existing[0] if existing else 'data'

DATA_DIR = discover_data_dir()
print(f"Data directory: {DATA_DIR}")
# =====================================================================
# =====================================================================
# Original local path:

import os
    os.path.join(os.getcwd(), 'Potsdam-GeoTif'),
    os.path.join(os.getcwd(), 'data'),
    os.path.join(os.getcwd(), 'PROJECT', 'Potsdam-GeoTif'),
    os.path.join(os.getcwd(), 'PROJECT', 'data'),
    os.path.join(os.path.dirname(os.getcwd()), 'Potsdam-GeoTif'),
    os.path.join(os.path.dirname(os.getcwd()), 'data'),
    os.getcwd()
]
# =====================================================================
# =====================================================================
# Original local path:

import os
    os.path.join(os.getcwd(), 'Potsdam-GeoTif'),
    os.path.join(os.getcwd(), 'data'),
    os.path.join(os.path.dirname(os.getcwd()), 'Potsdam-GeoTif'),
    os.path.join(os.path.dirname(os.getcwd()), 'data'),
    os.getcwd()
]
# =====================================================================

# اگر تعداد فایل‌ها کمتر از 5 برابر N_FOLDS است، برای demo تکثیر می‌کنیم
demo_files = all_files * max(1, (N_FOLDS * 2) // len(all_files) + 1)
demo_files = demo_files[:max(len(all_files), N_FOLDS * 2)]

kf  = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
arr = np.array(demo_files)
folds = [arr[idx].tolist() for _, idx in kf.split(arr)]

train_files = folds[0] + folds[1] + folds[2]
val_files   = folds[3]
test_files  = folds[4]

for i, f in enumerate(folds, 1):
    print(f'  Fold {i}: {len(f)} نمونه')
print(f'\n📚 Training (Fold 1+2+3): {len(train_files)} نمونه')
print(f'📋 Validation (Fold 4)  : {len(val_files)} نمونه')
print(f'🧪 Test (Fold 5)        : {len(test_files)} نمونه')

splits = {'train': train_files, 'val': val_files, 'test': test_files,
          'all_folds': folds}
splits_path = os.path.join(DATA_DIR, 'fold_splits.json')
with open(splits_path, 'w') as fp:
    json.dump(splits, fp, indent=2)
# # # print(f'\n✅ fold_splits.json ذخیره شد')# # print(f'\n✅ fold_splits.json ذخیره شد')# # print(f'\n✅ fold_splits.json ذخیره شد')# print(f'\n✅ fold_splits.json ذخیره شد')# # print(f'\n✅ fold_splits.json ذخیره شد')# print(f'\n✅ fold_splits.json ذخیره شد')# print(f'\n✅ fold_splits.json ذخیره شد')print(f'\n✅ fold_splits.json ذخیره شد')

### ✅ خلاصه مرحله ۱

- فایل‌های GeoTIFF پیدا شدند
- باندهای RGB، ارتفاع و برچسب تصویرسازی شدند
- توزیع کلاس‌ها محاسبه شد
- تقسیم‌بندی ۵-Fold انجام شد و در `fold_splits.json` ذخیره شد

**→ حالا به مرحله ۲ می‌رویم: آموزش مدل ساده CNN**


---
# 🧠 مرحله ۲ — مدل ساده CNN

## معماری مدل

یک شبکه عصبی کانولوشنی **fully-convolutional** (بدون downsampling) می‌سازیم:

```
Input (H×W×4)  →  Conv32 × 2  →  Conv64 × 2  →  Conv128 × 2  →  Conv1×1 → Softmax(6)
```

**ورودی:** فقط **۴ باند** — RGB + مادون قرمز (Infrared)

**خروجی:** نقشه احتمال به‌ازای هر پیکسل برای ۶ کلاس

**تابع خطا:** Categorical Cross-Entropy — برای مسائل چند کلاسه

**دوره‌های آموزش:** ۲۰ epoch


### ۲.۱ — توابع بارگذاری داده و Augmentation

**Data Augmentation** = تولید نسخه‌های مختلف از هر تصویر برای افزایش تنوع داده:
- چرخش تصادفی ۹۰ درجه (0، 90، 180، 270)
- آینه کردن افقی
- آینه کردن عمودی

این تکنیک از **overfitting** (حفظ داده به جای یادگیری) جلوگیری می‌کند.


In [ ]:
BATCH_SIZE_CNN = 2   # برای demo با 1 فایل کوچک است
EPOCHS_CNN     = 20
LR_CNN         = 1e-3
INPUT_CH_CNN   = 4   # RGB + IR

def load_sample(fp, use_all=False):
    """یک فایل GeoTIFF را می‌خواند و (feature, label_onehot) برمی‌گرداند."""
    with rasterio.open(fp) as src:
        d = src.read()
    n = 5 if use_all else 4
    X = d[:n].transpose(1, 2, 0).astype(np.float32)
    for c in range(X.shape[-1]):
        X[..., c] = normalize_band(X[..., c])
    y = tf.keras.utils.to_categorical(d[5].astype(np.int32), num_classes=NUM_CLASSES)
    return X, y

def augment(X, y):
    """Augmentation: چرخش تصادفی و آینه‌کاری."""
    if np.random.rand() > 0.5: X, y = X[:, ::-1, :], y[:, ::-1, :]
    if np.random.rand() > 0.5: X, y = X[::-1, :, :], y[::-1, :, :]
    k = np.random.randint(0, 4)
    return np.rot90(X, k).copy(), np.rot90(y, k).copy()

def make_ds(files, aug=False, bs=2, use_all=False):
    """یک tf.data.Dataset از لیست فایل‌ها می‌سازد."""
    Xs, ys = [], []
    for fp in files:
        X, y = load_sample(fp, use_all=use_all)
        if aug: X, y = augment(X, y)
        Xs.append(X); ys.append(y)
    ds = tf.data.Dataset.from_tensor_slices((np.stack(Xs), np.stack(ys)))
    if aug: ds = ds.shuffle(len(files), seed=SEED)
    return ds.batch(bs).prefetch(tf.data.AUTOTUNE)

print('✅ توابع بارگذاری داده آماده شدند')
print('✅ Augmentation: flip افقی + flip عمودی + چرخش 90°')

### ۲.۲ — ساخت مدل Simple CNN

هر **Encoder Block** شامل این لایه‌هاست:
- `Conv2D(k, 3×3)` → استخراج ویژگی با فیلترهای محلی
- `BatchNormalization` → پایدارسازی آموزش
- `ReLU` → تابع فعال‌سازی غیرخطی

در انتها `Conv2D(6, 1×1) + Softmax` → احتمال کلاس برای هر پیکسل


In [ ]:
def build_simple_cnn(in_ch=4, nc=6):
    """مدل Simple CNN برای تقسیم‌بندی معنایی."""
    inp = keras.Input(shape=(None, None, in_ch), name='input')
    x = inp
    for filters, name in [(32,'b1'), (64,'b2'), (128,'b3')]:
        x = layers.Conv2D(filters, 3, padding='same', activation='relu', name=f'{name}_c1')(x)
        x = layers.BatchNormalization(name=f'{name}_bn1')(x)
        x = layers.Conv2D(filters, 3, padding='same', activation='relu', name=f'{name}_c2')(x)
        x = layers.BatchNormalization(name=f'{name}_bn2')(x)
    out = layers.Conv2D(nc, 1, padding='same', activation='softmax', name='output')(x)
    model = keras.Model(inp, out, name='SimpleCNN')
    return model

cnn = build_simple_cnn(in_ch=INPUT_CH_CNN, nc=NUM_CLASSES)
cnn.compile(
    optimizer=keras.optimizers.Adam(LR_CNN),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print(f'✅ مدل ساخته شد: {cnn.name}')
print(f'📊 تعداد پارامترها: {cnn.count_params():,}')
cnn.summary()

### ۲.۳ — آموزش مدل CNN

مدل را روی **Training set** آموزش می‌دهیم و بهترین نسخه را ذخیره می‌کنیم.

- **ModelCheckpoint**: بهترین مدل (بر اساس val_accuracy) را ذخیره می‌کند
- **Epoch**: یک دور کامل از تمام داده‌های training

> ⏱️ با dataset کامل: آموزش ۲۰ epoch حدود ۱۰-۱۵ دقیقه روی GPU طول می‌کشد.


In [ ]:
print('📂 بارگذاری داده‌ها ...')
train_ds_cnn = make_ds(train_files, aug=True,  bs=BATCH_SIZE_CNN)
val_ds_cnn   = make_ds(val_files,   aug=False, bs=BATCH_SIZE_CNN)
test_ds_cnn  = make_ds(test_files,  aug=False, bs=BATCH_SIZE_CNN)

cb_cnn = [ModelCheckpoint(
    filepath=os.path.join(DATA_DIR, 'best_simple_model.keras'),
    monitor='val_accuracy', save_best_only=True, mode='max', verbose=0)]

print(f'\n🚀 شروع آموزش CNN برای {EPOCHS_CNN} epoch ...')
history_cnn = cnn.fit(
    train_ds_cnn, validation_data=val_ds_cnn,
    epochs=EPOCHS_CNN, callbacks=cb_cnn, verbose=1
)
print('✅ آموزش تمام شد!')

### ۲.۴ — نمودار آموزش CNN

نمودار **Loss** و **Accuracy** را برای آموزش و اعتبارسنجی رسم می‌کنیم.

- اگر **Training Loss** کاهش یابد ولی **Validation Loss** افزایش یابد → **Overfitting**
- برای حل Overfitting نیاز به داده بیشتر (dataset کامل) داریم


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ep = range(1, len(history_cnn.history['loss'])+1)

ax1.plot(ep, history_cnn.history['loss'],     label='Train Loss', color='royalblue', lw=2)
ax1.plot(ep, history_cnn.history['val_loss'], label='Val Loss',   color='darkorange', lw=2, ls='--')
ax1.set_title('Simple CNN — Loss', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(ep, history_cnn.history['accuracy'],     label='Train Acc', color='forestgreen', lw=2)
ax2.plot(ep, history_cnn.history['val_accuracy'], label='Val Acc',   color='crimson', lw=2, ls='--')
ax2.set_title('Simple CNN — Accuracy', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'step2_training_curves.png'), dpi=150)
plt.close()
display(Image(os.path.join(DATA_DIR, 'step2_training_curves.png')))

### ۲.۵ — ارزیابی روی Test Set

مدل را روی **Test Set** (Fold 5 که در حین آموزش استفاده نشده) ارزیابی می‌کنیم.


In [ ]:
best_cnn = keras.models.load_model(os.path.join(DATA_DIR, 'best_simple_model.keras'))
loss_cnn, acc_cnn = best_cnn.evaluate(test_ds_cnn, verbose=0)

print('=' * 40)
print('  📊 نتایج Simple CNN روی Test Set')
print(f'  Loss    : {loss_cnn:.4f}')
print(f'  Accuracy: {acc_cnn*100:.2f}%')
print('=' * 40)
print()
print(f'  آخرین Train Accuracy: {history_cnn.history["accuracy"][-1]*100:.2f}%')
print(f'  بهترین Val Accuracy : {max(history_cnn.history["val_accuracy"])*100:.2f}%')

### ✅ خلاصه مرحله ۲

- مدل Simple CNN با ۳ بلاک encoder ساخته شد
- با Augmentation (flip + rotation) آموزش دیده شد
- نمودار loss و accuracy رسم شد
- بهترین مدل در `best_simple_model.keras` ذخیره شد

**→ حالا به مرحله ۳ می‌رویم: مدل U-Net پیشرفته‌تر**


---
# 🏗️ مرحله ۳ — مدل U-Net (Encoder-Decoder)

## چرا U-Net؟

مدل Simple CNN ضعف اساسی دارد: **جزئیات مکانی را از دست می‌دهد**.

**U-Net** این مشکل را با معماری خاص خود حل می‌کند:

```
Input
  ↓ Encoder Block 1 (32) ─────────────────────────────┐ Skip
  ↓ MaxPool                                            |
  ↓ Encoder Block 2 (64) ─────────────────────┐ Skip  |
  ↓ MaxPool                                    |       |
  ↓ Encoder Block 3 (128) ──────────┐ Skip    |       |
  ↓ MaxPool                         |         |       |
  ↓ Encoder Block 4 (256) ──┐ Skip  |         |       |
  ↓ MaxPool                  |       |         |       |
  ↓ Bottleneck (512)         |       |         |       |
  ↑ UpSample + Concat ───────┘       |         |       |
  ↑ Decoder Block 4 (256)            |         |       |
  ↑ UpSample + Concat ───────────────┘         |       |
  ↑ Decoder Block 3 (128)                      |       |
  ↑ UpSample + Concat ─────────────────────────┘       |
  ↑ Decoder Block 2 (64)                               |
  ↑ UpSample + Concat ─────────────────────────────────┘
  ↑ Decoder Block 1 (32)
  ↓ Output Conv 1×1 + Softmax(6)
```

**Skip Connections** (پیوندهای میانبر) → اطلاعات مکانی دقیق از encoder به decoder منتقل می‌شود.

**ورودی:** ۵ باند — RGB + IR + ارتفاع


### ۳.۱ — ساخت معماری U-Net

هر **conv_block** شامل دو Conv2D با BatchNorm است.
در بخش Decoder، ابتدا **UpSampling** (دو برابر کردن اندازه) انجام می‌شود،
سپس **Concatenate** با feature map متناظر encoder (skip connection).


In [ ]:
def conv_block(x, f, name):
    x = layers.Conv2D(f, 3, padding='same', activation='relu', name=f'{name}_c1')(x)
    x = layers.BatchNormalization(name=f'{name}_bn1')(x)
    x = layers.Conv2D(f, 3, padding='same', activation='relu', name=f'{name}_c2')(x)
    x = layers.BatchNormalization(name=f'{name}_bn2')(x)
    return x

def build_unet(in_ch=5, nc=6, f=32):
    """U-Net Encoder-Decoder برای تقسیم‌بندی معنایی."""
    inp = keras.Input(shape=(None, None, in_ch), name='input')
    # Encoder
    e1 = conv_block(inp, f,    'enc1'); p1 = layers.MaxPooling2D(2, name='pool1')(e1)
    e2 = conv_block(p1,  f*2,  'enc2'); p2 = layers.MaxPooling2D(2, name='pool2')(e2)
    e3 = conv_block(p2,  f*4,  'enc3'); p3 = layers.MaxPooling2D(2, name='pool3')(e3)
    e4 = conv_block(p3,  f*8,  'enc4'); p4 = layers.MaxPooling2D(2, name='pool4')(e4)
    # Bottleneck
    b  = conv_block(p4,  f*16, 'bottleneck')
    # Decoder + Skip Connections
    d4 = conv_block(layers.Concatenate()([layers.UpSampling2D(2)(b),  e4]), f*8,  'dec4')
    d3 = conv_block(layers.Concatenate()([layers.UpSampling2D(2)(d4), e3]), f*4,  'dec3')
    d2 = conv_block(layers.Concatenate()([layers.UpSampling2D(2)(d3), e2]), f*2,  'dec2')
    d1 = conv_block(layers.Concatenate()([layers.UpSampling2D(2)(d2), e1]), f,    'dec1')
    out = layers.Conv2D(nc, 1, padding='same', activation='softmax', name='output')(d1)
    return keras.Model(inp, out, name='UNet')

unet = build_unet(in_ch=5, nc=NUM_CLASSES, f=32)
unet.compile(
    optimizer=keras.optimizers.Adam(1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print(f'✅ مدل ساخته شد: {unet.name}')
print(f'📊 تعداد پارامترها: {unet.count_params():,}')
unet.summary()

### ۳.۲ — آموزش U-Net

آموزش U-Net با learning rate کمتر (`1e-4`) انجام می‌شود:
- Learning rate پایین‌تر → آموزش پایدارتر برای معماری‌های بزرگ
- Batch size کوچک‌تر (2 به جای 16) → به خاطر حجم بیشتر پارامترها

> ⏱️ با dataset کامل و GPU: حدود ۱۰-۱۵ دقیقه


In [ ]:
EPOCHS_UNET = 20
BATCH_SIZE_UNET = 2

print('📂 بارگذاری داده‌ها برای U-Net (5 باند) ...')
train_ds_u = make_ds(train_files, aug=True,  bs=BATCH_SIZE_UNET, use_all=True)
val_ds_u   = make_ds(val_files,   aug=False, bs=BATCH_SIZE_UNET, use_all=True)
test_ds_u  = make_ds(test_files,  aug=False, bs=BATCH_SIZE_UNET, use_all=True)

cb_unet = [ModelCheckpoint(
    filepath=os.path.join(DATA_DIR, 'best_unet_model.keras'),
    monitor='val_accuracy', save_best_only=True, mode='max', verbose=0)]

print(f'\n🚀 شروع آموزش U-Net برای {EPOCHS_UNET} epoch ...')
history_unet = unet.fit(
    train_ds_u, validation_data=val_ds_u,
    epochs=EPOCHS_UNET, callbacks=cb_unet, verbose=1
)
print('✅ آموزش تمام شد!')

### ۳.۳ — نمودار آموزش U-Net


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ep = range(1, len(history_unet.history['loss'])+1)

ax1.plot(ep, history_unet.history['loss'],     label='Train Loss', color='royalblue', lw=2)
ax1.plot(ep, history_unet.history['val_loss'], label='Val Loss',   color='darkorange', lw=2, ls='--')
ax1.set_title('U-Net — Loss', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(ep, history_unet.history['accuracy'],     label='Train Acc', color='forestgreen', lw=2)
ax2.plot(ep, history_unet.history['val_accuracy'], label='Val Acc',   color='crimson', lw=2, ls='--')
ax2.set_title('U-Net — Accuracy', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'step3_training_curves.png'), dpi=150)
plt.close()
display(Image(os.path.join(DATA_DIR, 'step3_training_curves.png')))

### ۳.۴ — ارزیابی و تصویرسازی پیش‌بینی

این مهم‌ترین بخش مرحله ۳ است: **نمایش بصری پیش‌بینی U-Net**

چهار پنجره نشان می‌دهیم:
1. تصویر RGB اصلی
2. باند ارتفاع
3. برچسب واقعی (Ground Truth)
4. پیش‌بینی U-Net


In [ ]:
best_unet = keras.models.load_model(os.path.join(DATA_DIR, 'best_unet_model.keras'))
loss_u, acc_u = best_unet.evaluate(test_ds_u, verbose=0)

print('=' * 40)
print('  📊 نتایج U-Net روی Test Set')
print(f'  Loss    : {loss_u:.4f}')
print(f'  Accuracy: {acc_u*100:.2f}%')
print('=' * 40)

# پیش‌بینی روی یک تایل نمونه
test_fp = test_files[0]
X5, _, = load_sample(test_fp, use_all=True), None
X5 = load_sample(test_fp, use_all=True)[0]

with rasterio.open(test_fp) as src:
    raw = src.read()

gt_label   = raw[5].astype(np.int32)
pred_logit = best_unet.predict(np.expand_dims(X5, 0), verbose=0)[0]
pred_label = np.argmax(pred_logit, axis=-1)

rgb_v = np.stack([normalize_band(raw[0].astype(np.float32)),
                  normalize_band(raw[1].astype(np.float32)),
                  normalize_band(raw[2].astype(np.float32))], axis=-1)
elev_v  = normalize_band(raw[4].astype(np.float32))
gt_rgb  = label_to_rgb(gt_label,   CLASS_COLORS)
pred_rgb= label_to_rgb(pred_label, CLASS_COLORS)

patches = [mpatches.Patch(color=[c/255 for c in CLASS_COLORS[i]], label=CLASS_NAMES[i])
           for i in range(NUM_CLASSES)]

fig, axes = plt.subplots(1, 4, figsize=(22, 6))
fig.suptitle('U-Net: پیش‌بینی در برابر واقعیت', fontsize=13, fontweight='bold')
axes[0].imshow(rgb_v);   axes[0].set_title('تصویر RGB');        axes[0].axis('off')
im = axes[1].imshow(elev_v, cmap='terrain'); axes[1].set_title('ارتفاع'); axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
axes[2].imshow(gt_rgb);  axes[2].set_title('Ground Truth');     axes[2].axis('off')
axes[2].legend(handles=patches, loc='lower right', fontsize=6.5, framealpha=0.9)
axes[3].imshow(pred_rgb);axes[3].set_title('پیش‌بینی U-Net');  axes[3].axis('off')
axes[3].legend(handles=patches, loc='lower right', fontsize=6.5, framealpha=0.9)

plt.tight_layout()
plt.savefig(os.path.join(DATA_DIR, 'step3_prediction_visualization.png'), dpi=150, bbox_inches='tight')
plt.close()
display(Image(os.path.join(DATA_DIR, 'step3_prediction_visualization.png')))

---
# 📊 مقایسه نهایی مدل‌ها

در این بخش نتایج هر دو مدل را با هم مقایسه می‌کنیم.


In [ ]:
print('\n' + '=' * 60)
print('  📊 مقایسه نهایی مدل‌ها')
print('=' * 60)
print(f'{"مدل":<15} {"باندها":<10} {"Test Loss":>12} {"Test Acc":>10}')
print('-' * 50)
print(f'{"Simple CNN":<15} {"4 (RGB+IR)":<10} {loss_cnn:>12.4f} {acc_cnn*100:>9.2f}%')
print(f'{"U-Net":<15} {"5 (+Elev)":<10} {loss_u:>12.4f} {acc_u*100:>9.2f}%')
print('=' * 60)

winner = 'U-Net' if loss_u < loss_cnn else 'Simple CNN'
print(f'\n🏆 مدل برتر بر اساس Test Loss: {winner}')
print(f'   اختلاف Loss: {abs(loss_cnn - loss_u):.4f}')

---
# 🎓 جمع‌بندی و نتیجه‌گیری

## آنچه یاد گرفتیم:

### مرحله ۱ — داده
- فایل‌های GeoTIFF دارای ۶ باند هستند که باند آخر برچسب کلاس است
- تقسیم‌بندی K-Fold از خطای ارزیابی بی‌طرف اطمینان می‌دهد

### مرحله ۲ — Simple CNN
- مدل‌های fully-convolutional می‌توانند مستقیماً پیش‌بینی پیکسل به پیکسل کنند
- Overfitting با dataset کوچک مسئله جدی است

### مرحله ۳ — U-Net
- Skip connections اطلاعات مکانی را حفظ می‌کنند
- U-Net برای تقسیم‌بندی معنایی تصاویر زمینی/ماهواره‌ای استاندارد صنعتی است

## گام‌های بعدی با Dataset کامل:
1. `DATA_DIR` را به مسیر dataset کامل تغییر دهید
2. این notebook را از ابتدا اجرا کنید
3. انتظار داریم Test Accuracy به **بیش از ۸۰٪** برسد

---
> 📄 گزارش کامل LaTeX: `project_overview.tex` → آماده برای Overleaf
